# CNN baseline — U-Net + ResNet50 on Sen1Floods11 (issue #19)

Scale: 252 hand-labelled training chips, ~31 iterations/epoch at batch 8.

Issue #19 expects water IoU around **0.55–0.65**, below the published figure.
That shortfall is the expected reproduction gap.

## 1. Code and dependencies

In [ ]:
!git clone -q https://github.com/cs59300-deep-learning/nisar-flood-mapping.git
%cd nisar-flood-mapping
!pip install -q segmentation-models-pytorch timm rasterio pyyaml

## 2. Check what hardware you landed on

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    major = torch.cuda.get_device_capability(0)[0]
    print(f"{name} | {vram:.0f} GB | compute capability {major}.x")
    print("bfloat16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No CUDA. MPS:", torch.backends.mps.is_available(), "- expect a slow run.")

### Tuning for a larger GPU

The defaults target a 16 GB T4. On an A100 or H100 the code runs unchanged, but
two things are worth knowing:

- **It will not be proportionally faster.** The model is small (32M parameters)
  and the dataset tiny, so on a big GPU the bottleneck moves from compute to
  reading GeoTIFFs off disk. Raise `data.workers` before raising batch size.
- **Resist a much larger batch.** 252 chips at batch 8 is only ~31 optimiser
  steps per epoch. At batch 32 it is 7, which is too few to learn well without
  also changing the learning rate and epoch count — and then the run is no
  longer comparable with the transformer arm (#20).

On Ampere and newer, enabling TF32 is free speed with no accuracy cost here.

In [ ]:
# Safe on A100/H100/4090 and ignored elsewhere.
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    torch.set_float32_matmul_precision("high")   # TF32 for matmuls
    print("TF32 enabled")

## 3. Data

The download fetches 446 chips plus labels from Google Cloud Storage and takes
a while. If Sen1Floods11 is already mirrored as a dataset on your host, attach
it and point `--root` at that path instead.

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli download --root data/sen1floods11
!PYTHONPATH=src python -m nisar_flood.cli check    --root data/sen1floods11

`check` should print 252 / 89 / 90 / 15 for train / val / test / bolivia, no
missing files, and inputs inside [0, 1]. **Stop here if it does not.**

## 4. Train

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli train --config configs/unet_resnet50.yaml --seed 0

Writes to `runs/unet_resnet50/seed0/`: `history.csv`, `curves.png`, `best.pt`,
`last.pt`, `summary.json`. Checkpoint selection uses the validation image-wise
mean water IoU, matching the Sen1Floods11 definition.

## 5. Evaluate on the held-out splits

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli evaluate \
    --checkpoint runs/unet_resnet50/seed0/best.pt

`test` is the headline number for #19. `bolivia` is Sen1Floods11's
out-of-distribution split, reported separately — it is the closest thing in the
source domain to the transfer question E4 asks about NISAR, so it earns a line
in the report.

## 6. Keep the results

In [ ]:
import shutil, pathlib
# Many hosted runtimes are wiped when the session ends. Copy the small artefacts
# somewhere persistent: /kaggle/working on Kaggle, your Drive on Colab, or any
# local path. Not the checkpoints - they are ~130 MB each.
DEST = pathlib.Path("artifacts/unet_resnet50_seed0")   # change for your host
DEST.mkdir(parents=True, exist_ok=True)
for name in ["history.csv", "curves.png", "summary.json", "eval.json"]:
    src = pathlib.Path("runs/unet_resnet50/seed0") / name
    if src.exists():
        shutil.copy(src, DEST / name)
print(sorted(p.name for p in DEST.iterdir()))

Then put `curves.png`, `history.csv` and `eval.json` into `reports/` — those are
#19's acceptance criteria. **Do not commit `best.pt`**: checkpoints go to the
Hugging Face dataset repo, per `docs/data_sources.md`.